### Extracting Weather Data

In [ ]:
pip install meteostat

In [1]:
from meteostat import Point, Daily
from datetime import datetime

In [2]:
import pandas as pd

In [3]:
# Define parameters for data extraction
start_date = datetime(2019, 1, 1)  # Start of the time range
end_date = datetime(2024, 12, 31)  # End of the time range

In [4]:
# Define a central point in Germany (approximate latitude/longitude)
germany_center = Point(51.1657, 10.4515)  # Latitude/Longitude of Germany's centroid

In [5]:
# Fetch daily weather data for Germany's central grid point
data = Daily(germany_center, start_date, end_date)
data = data.fetch()

In [6]:
# Calculate HDD and CDD using base temperature of 18°C
data['HDD'] = data['tavg'].apply(lambda x: max(18 - x, 0) if not pd.isnull(x) else None)
data['CDD'] = data['tavg'].apply(lambda x: max(x - 18, 0) if not pd.isnull(x) else None)

In [7]:
# Convert the data to a DataFrame and format the Date column
daily_data = data.reset_index()
daily_data['time'] = daily_data['time'].dt.strftime('%d.%m.%Y')  # Format the date to dd.mm.yyyy

In [8]:
# Rename columns for clarity
daily_data.rename(columns={
    'time': 'date',
    'tavg': 'Mean_Temperature',
    'prcp': 'Precipitation',
    'wspd': 'Wind_Speed',
}, inplace=True)

In [9]:
output_path = "Weather_Data_Germany_Daily_2019_2024.csv"  # Replace with your desired path
daily_data.to_csv(output_path, index=False)
print(f"Daily weather data saved to: {output_path}")

Daily weather data saved to: Weather_Data_Germany_Daily_2019_2024.csv


In [10]:
# Display the first few rows of the daily data
print(daily_data.head())

         date  Mean_Temperature  tmin  tmax  Precipitation  snow   wdir  \
0  01.01.2019               6.2   0.5   8.1            3.1   0.0  267.0   
1  02.01.2019               2.3  -0.5   3.6            0.0  10.0  319.0   
2  03.01.2019              -0.1  -3.4   1.6            0.0   0.0  292.0   
3  04.01.2019               2.1   0.1   4.7            0.3   0.0  273.0   
4  05.01.2019               6.4   4.6   8.0            0.6   0.0  284.0   

   Wind_Speed  wpgt    pres   tsun   HDD  CDD  
0        20.2  63.0  1024.9    0.0  11.8  0.0  
1        17.3  49.7  1030.1  204.0  15.7  0.0  
2         8.6  22.0  1036.8  138.0  18.1  0.0  
3        13.7  34.2  1033.2    0.0  15.9  0.0  
4        16.2  38.5  1024.7    0.0  11.6  0.0  


In [11]:
# Select the required columns
filtered_data = daily_data[[
    'date',
    'Mean_Temperature',  # Average daily temperature
    'HDD',               # Heating Degree Days
    'CDD',               # Cooling Degree Days
    'Precipitation',     # Precipitation in mm
    'Wind_Speed'         # Average wind speed in m/s
]]

In [12]:
# Overwrite the same file with the filtered data
filtered_data.to_csv(output_path, index=False)

In [13]:
print(f"Filtered data saved to the same file: {output_path}")

Filtered data saved to the same file: Weather_Data_Germany_Daily_2019_2024.csv


### Extracting solar radiation data

In [8]:
pip install cdsapi>=0.7.2

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install netCDF4 h5netcdf

In [ ]:
pip show cdsapi

In [3]:
import cdsapi
import xarray as xr
import pandas as pd

In [15]:
# Initialize the CDS API client
client = cdsapi.Client()

2025-01-14 22:24:16,031 INFO [2024-09-28T00:00:00] **Welcome to the New Climate Data Store (CDS)!** This new system is in its early days of full operations and still undergoing enhancements and fine tuning. Some disruptions are to be expected. Your 
[feedback](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) is key to improve the user experience on the new CDS for the benefit of everyone. Thank you.
2025-01-14 22:24:16,034 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-01-14 22:24:16,035 INFO [2024-09-16T00:00:00] Remember that you need to have an ECMWF account to use the new CDS. **Your old CDS credentials will not work in new CDS!**
2025-01-14 22:24:16,039 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using t

In [16]:
# Define the dataset and request parameters
dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": "reanalysis",
    "variable": ["surface_solar_radiation_downwards"],  # Solar radiation parameter
    "area": [55, 5, 47, 15],  # Germany bounding box (N, W, S, E)
    "day": [f"{i:02d}" for i in range(1, 32)],  # All days of the month
    "month": [f"{i:02d}" for i in range(1, 13)],  # All months
    "time": [f"{i:02d}:00" for i in range(24)],  # All hours of the day
    "format": "netcdf",
    "year": ["2019", "2020", "2021", "2022", "2023", "2024"],  # Requested years
}

In [5]:
filename = "solar_radiation_2019_2024.nc"

In [18]:
filename = "solar_radiation_2019_2024.nc"
print(f"Requesting data for 2019-2024...")
client.retrieve(dataset, request).download(filename)
print(f"Downloaded: {filename}")

Requesting data for 2019-2024...


2025-01-14 22:24:28,880 INFO [2025-01-09T00:00:00] Please be aware that ERA5 data from 1st January 2025 was degraded and is being corrected. Watch the [Forum announcement](https://forum.ecmwf.int/t/era5-data-from-1st-january-2025-was-degraded-and-is-being-corrected/10689) for updates.
2025-01-14 22:24:28,883 INFO Request ID is d60232f2-ed94-429b-a6ea-e895e748e8f9
2025-01-14 22:24:28,972 INFO status has been updated to accepted
2025-01-14 22:24:34,013 INFO status has been updated to running
2025-01-14 23:44:58,476 INFO status has been updated to successful


89e7a788f583b77bda8b05d96ab63247.nc:   0%|          | 0.00/81.6M [00:00<?, ?B/s]

Downloaded: solar_radiation_2019_2024.nc


In [6]:
print(f"Processing file: {filename}")

Processing file: solar_radiation_2019_2024.nc


In [7]:
# Load the NetCDF file
ds = xr.open_dataset(filename)

In [8]:
# Print the dataset structure
print(ds)

<xarray.Dataset>
Dimensions:     (valid_time: 52608, latitude: 33, longitude: 41)
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 2019-01-01 ... 2024-12-31T23:00:00
  * latitude    (latitude) float64 55.0 54.75 54.5 54.25 ... 47.5 47.25 47.0
  * longitude   (longitude) float64 5.0 5.25 5.5 5.75 ... 14.25 14.5 14.75 15.0
    expver      (valid_time) object ...
Data variables:
    ssrd        (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-01-14T22:40 GRIB to CDM+CF via cfgrib-0.9.1...


In [9]:
# Compute spatial mean first (reduce size and complexity)
spatial_mean = ds['ssrd'].mean(dim=['latitude', 'longitude'])

In [10]:
# Aggregate hourly data into daily sums
daily_radiation = (
    spatial_mean
    .groupby(spatial_mean['valid_time'].dt.date)  # Group by valid_time date
    .sum()  # Sum over hourly values for each day
    / 1e6  # Convert J/m² to MJ/m²
)

In [11]:
# Convert to a DataFrame for easier handling
df = daily_radiation.to_dataframe().reset_index()

In [12]:
# Rename columns for clarity
df.rename(columns={'valid_time': 'date', 'ssrd': 'Solar_Radiation_MJ_m2'}, inplace=True)

In [13]:
# Convert Date to dd.mm.yyyy format
df['date'] = pd.to_datetime(df['date']).dt.strftime('%d.%m.%Y')

In [14]:
if 'number' in df.columns:
    df.drop(columns=['number'], inplace=True)

In [15]:
output_path = "solar_radiation_daily_2019_2024.csv"
df.to_csv(output_path, index=False)
print(f"Daily solar radiation data saved to: {output_path}")

Daily solar radiation data saved to: solar_radiation_daily_2019_2024.csv


In [16]:
print(df.head())

         date  Solar_Radiation_MJ_m2
0  01.01.2019               1.850080
1  02.01.2019               3.267914
2  03.01.2019               3.021612
3  04.01.2019               1.804020
4  05.01.2019               0.970135
